In [ ]:
!pip install pymupdf4llm
!pip install chromadb
!pip install litellm
!pip install -U langchain langchain-community langchain-core langchain-text-splitters langgraph langsmith

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
import hashlib
import re
import pickle
import time
from typing import TypedDict, List, Dict, Any, Annotated

import collections
import pymupdf4llm
import chromadb
import networkx as nx
from chromadb.utils import embedding_functions
from litellm import completion
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END
from google.colab import userdata, drive

_call_timestamps = collections.deque()
RATE_LIMIT_CALLS = 4      # stay under the real 5/min cap with margin
RATE_LIMIT_WINDOW = 62    # seconds, slightly over 60 for safety

def throttled_completion(**kwargs):
    now = time.time()
    while _call_timestamps and now - _call_timestamps[0] > RATE_LIMIT_WINDOW:
        _call_timestamps.popleft()

    if len(_call_timestamps) >= RATE_LIMIT_CALLS:
        wait = RATE_LIMIT_WINDOW - (now - _call_timestamps[0]) + 1
        print(f"⏳ Rate limit guard: API cap reached, waiting {wait:.0f}s...")
        time.sleep(wait)
        now = time.time()
        while _call_timestamps and now - _call_timestamps[0] > RATE_LIMIT_WINDOW:
            _call_timestamps.popleft()

    _call_timestamps.append(now)
    return completion(**kwargs)

# ---------------------------------------------------------
# 0. CONFIG & ONTOLOGY
# ---------------------------------------------------------
MODEL_NAME = "gemini/gemini-3.5-flash"
EMBED_MODEL = "all-MiniLM-L6-v2"

EQUIPMENT_HAZARDS = {
    "PUMP-14": ["Flammable Gas", "High Pressure", "Mechanical"],
    "COKE-OVEN-02": ["Thermal/Heat", "Toxic/Asphyxiant Gas", "Confined Space"],
    "BOILER-B1": ["High Pressure", "Thermal/Heat"],
    "VALVE-102": ["High Pressure", "Chemical Leak"],
    "CONVEYOR-C3": ["Mechanical", "Fire/Dust"],
    "COMPRESSOR-A": ["High Pressure", "Mechanical"],
    "STORAGE-TANK-T5": ["Confined Space", "Flammable Gas"],
    "TRANSFORMER-TR1": ["Electrical", "Thermal/Heat"],
    "EXHAUST-FAN-F2": ["Mechanical"],
    "CRANE-OH-01": ["Mechanical"],
}
REGULATORY_HAZARDS = {
    "factories_act_1948.pdf": ["Mechanical", "Thermal/Heat", "High Pressure", "Confined Space", "Chemical Leak"],
    "dgms_2020_04_gas_hazards.pdf": ["Flammable Gas", "Toxic/Asphyxiant Gas"],
    "dgms_2017_loto_procedures.pdf": ["Electrical", "Mechanical", "High Pressure"],
    "dgms_2020_05_oil_gas_safety.pdf": ["High Pressure", "Flammable Gas"],
}
KNOWN_EQUIPMENTS = list(EQUIPMENT_HAZARDS.keys())
KNOWN_HAZARDS = list(set(h for lst in EQUIPMENT_HAZARDS.values() for h in lst))

# ---------------------------------------------------------
# 1. DRIVE / PATH SETUP
# ---------------------------------------------------------
def ensure_drive_mounted(mount_point="/content/drive"):
    if not os.path.ismount(mount_point):
        print("🔗 Mounting Google Drive...")
        drive.mount(mount_point)
    if not os.path.ismount(mount_point):
        raise RuntimeError(f"❌ Google Drive did NOT mount at {mount_point}.")
    print(f"✅ Google Drive verified as mounted at {mount_point}")

def verify_base_path(base_path):
    reg_dir = os.path.join(base_path, "regulatory")
    if not os.path.isdir(reg_dir):
        raise RuntimeError(f"❌ {reg_dir} does not exist.")
    pdfs = [f for f in os.listdir(reg_dir) if f.endswith(".pdf")]
    if not pdfs:
        raise RuntimeError(f"❌ {reg_dir} has no PDFs.")
    print(f"✅ base_path verified — {len(pdfs)} PDF(s) found")

def hash_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

# ---------------------------------------------------------
# 2. LOAD VECTOR STORE & GRAPH
# ---------------------------------------------------------
def ingest_regulatory_pdf(pdf_path, chunk_offset=0):
    md_text = pymupdf4llm.to_markdown(pdf_path)
    parts = re.compile(r'(\n\*\*\d+[A-Z]?\.\s+[^\*]+\*\*)').split(md_text)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
    chunks, idx = [], chunk_offset
    if parts[0].strip():
        for t in splitter.split_text(parts[0].strip()):
            chunks.append({"text": t, "metadata": {"source": os.path.basename(pdf_path),
                          "doc_type": "regulatory", "section_title": "Preamble / TOC",
                          "chunk_id": f"chunk_{idx}"}})
            idx += 1
    for i in range(1, len(parts), 2):
        header, body = parts[i].strip(), parts[i + 1].strip()
        title = header.replace('**', '').split('.—')[0].split('—')[0].strip()
        for t in splitter.split_text(f"{header}\n{body}"):
            chunks.append({"text": t, "metadata": {"source": os.path.basename(pdf_path),
                          "doc_type": "regulatory", "section_title": title,
                          "chunk_id": f"chunk_{idx}"}})
            idx += 1
    return chunks, idx

def ingest_synthetic_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, dict):
        return []
    chunks = []
    for cat in ['maintenance_logs', 'incident_reports', 'work_permits']:
        for item in data.get(cat, []):
            rid = (item.get('log_id') or item.get('incident_id') or item.get('permit_id')
                   or hashlib.md5(str(item).encode()).hexdigest()[:8])
            text = "\n".join([f"Record Type: {cat}"] + [f"{k}: {v}" for k, v in item.items()])
            chunks.append({"text": text, "metadata": {
                "source": os.path.basename(json_path), "doc_type": cat,
                "equipment_id": item.get("equipment_id", "UNKNOWN"),
                "record_id": rid, "chunk_id": f"{cat}_{rid}"}})
    return chunks

def load_vector_store(data_dirs, cache_path, chroma_path):
    manifest_path = cache_path.replace('.json', '_manifest.json')
    if os.path.exists(manifest_path) and os.path.exists(cache_path):
        manifest = json.load(open(manifest_path, encoding='utf-8'))
        all_chunks = json.load(open(cache_path, encoding='utf-8'))
    else:
        manifest, all_chunks = {}, []

    new_chunks, needs_update, counter = [], False, len(all_chunks)
    for d in data_dirs:
        if not os.path.exists(d):
            continue
        for f in sorted(os.listdir(d)):
            fp = os.path.join(d, f)
            if (f.endswith('.pdf') or (f.endswith('.json') and 'synthetic' in d)) and os.path.getsize(fp) > 0:
                h = hash_file(fp)
                if manifest.get(f) != h:
                    needs_update, manifest[f] = True, h
                    if f.endswith('.pdf'):
                        c, counter = ingest_regulatory_pdf(fp, counter)
                        new_chunks.extend(c)
                    else:
                        new_chunks.extend(ingest_synthetic_json(fp))

    emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
    client = chromadb.PersistentClient(path=chroma_path)
    collection = client.get_or_create_collection(name="industrial_brain_v2", embedding_function=emb_fn)

    if needs_update:
        all_chunks.extend(new_chunks)
        json.dump(all_chunks, open(cache_path, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
        json.dump(manifest, open(manifest_path, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
        collection.upsert(documents=[d["text"] for d in new_chunks],
                          metadatas=[d["metadata"] for d in new_chunks],
                          ids=[d["metadata"]["chunk_id"] for d in new_chunks])
        print(f"💾 {len(new_chunks)} new chunks embedded and stored.")
    elif collection.count() == 0 and all_chunks:
        collection.upsert(documents=[d["text"] for d in all_chunks],
                          metadatas=[d["metadata"] for d in all_chunks],
                          ids=[d["metadata"]["chunk_id"] for d in all_chunks])
        print("💾 Re-hydrated vector store from cache.")
    else:
        print("✅ Vector store already current.")
    print(f"📊 Total chunks in DB: {collection.count()}")
    return collection

def load_knowledge_graph(json_path, graph_path):
    manifest_path = graph_path.replace('.pkl', '_manifest.json')
    current_hash = hash_file(json_path) if os.path.exists(json_path) else None
    if os.path.exists(graph_path) and os.path.exists(manifest_path):
        manifest = json.load(open(manifest_path))
        if manifest.get('industrial_records.json') == current_hash:
            print("✅ Knowledge graph loaded from cache.")
            return pickle.load(open(graph_path, 'rb'))
    raise RuntimeError("❌ No valid cached knowledge_graph.pkl found. Run Phase 3 first.")

# ---------------------------------------------------------
# 3. GRAPH QUERY HELPERS
# ---------------------------------------------------------
def query_knowledge_graph(G, entity_id, relation=None):
    results = []
    if entity_id not in G:
        return results
    for u, v, d in G.out_edges(entity_id, data=True):
        if relation is None or d.get('label') == relation:
            results.append((entity_id, d.get('label'), v))
    for u, v, d in G.in_edges(entity_id, data=True):
        if relation is None or d.get('label') == relation:
            results.append((u, d.get('label'), entity_id))
    return results

def get_regulations_for_equipment(G, equipment_id):
    regs = set()
    if equipment_id not in G:
        return regs
    hazards = [v for u, v, d in G.out_edges(equipment_id, data=True) if d.get('label') == 'HAS_HAZARD']
    for haz in hazards:
        for u, v, d in G.in_edges(haz, data=True):
            if d.get('label') == 'APPLIES_TO':
                regs.add((u, haz))
    locations = [v for u, v, d in G.out_edges(equipment_id, data=True) if d.get('label') == 'LOCATED_IN']
    for loc in locations:
        for u, v, d in G.in_edges(loc, data=True):
            if d.get('label') == 'APPLIES_TO_ALL':
                regs.add((u, loc))
    return regs

def extract_graph_context(G, resolved_entities):
    facts = []
    for entity in resolved_entities:
        if entity not in G:
            continue
        node_type = G.nodes[entity].get('type')
        if node_type == 'Equipment':
            for u, rel, v in query_knowledge_graph(G, entity):
                facts.append(f"GRAPH FACT: [{u}] -> {rel} -> [{v}] [graph:{rel}]")
            for reg, target in get_regulations_for_equipment(G, entity):
                if G.nodes[target].get('type') == 'HazardType':
                    facts.append(f"GRAPH FACT: [{reg}] applies to hazard [{target}] present in [{entity}] [graph:APPLIES_TO]")
                else:
                    facts.append(f"GRAPH FACT: [{reg}] applies broadly to plant section [{target}] where [{entity}] is located [graph:APPLIES_TO_ALL]")
        elif node_type == 'HazardType':
            equipments = [u for u, v, d in G.in_edges(entity, data=True) if d.get('label') == 'HAS_HAZARD']
            for eq in equipments:
                facts.append(f"GRAPH FACT: Equipment [{eq}] has hazard [{entity}] [graph:HAS_HAZARD]")
                for inc in [u for u, v, d in G.in_edges(eq, data=True) if d.get('label') == 'INVOLVED']:
                    facts.append(f"GRAPH FACT: Incident [{inc}] INVOLVED Equipment [{eq}] [graph:INVOLVED]")
    return list(set(facts))

# ---------------------------------------------------------
# 4. RETRIEVAL, GENERATION, VERIFICATION
# ---------------------------------------------------------
def retrieve_chunks(collection, query, doc_type=None, n_results=5):
    results = collection.query(query_texts=[query], n_results=n_results,
                                where={"doc_type": doc_type} if doc_type else None)
    ids = results['ids'][0]
    formatted = [f"--- ID: {cid} (Source: {m['source']}) ---\n{txt}"
                 for cid, m, txt in zip(ids, results['metadatas'][0], results['documents'][0])]
    return "\n\n".join(formatted), ids

def verify_citations(answer, context_str):
    prompt = f"""Check whether every bracketed citation (e.g., [chunk_12] or [graph:ISSUED_FOR]) in ANSWER\nis genuinely supported by its matching source in CONTEXT. A citation naming a source that only\nlists a section number or node, without stating the actual fact claimed, counts as unsupported.\n\nCONTEXT:\n{context_str}\n\nANSWER:\n{answer}\n\nRespond in strict JSON only, no markdown fences:\n{{"grounded": true/false, "unsupported_claims": ["..."]}}"""
    resp = completion(model=MODEL_NAME, messages=[{"role": "user", "content": prompt}],
                        temperature=0.0, api_key=userdata.get('GEMINI_API_KEY'))
    raw = re.sub(r'^```json\s*|\s*```$', '', resp.choices[0].message.content.strip())
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"grounded": None, "unsupported_claims": [f"verifier non-JSON: {raw[:200]}"]}

def ask_fast(collection, query, query_cache_path, doc_type=None, force_refresh=False):
    cache = load_query_cache(query_cache_path)
    key = hashlib.md5(f"{query}::{doc_type}".encode()).hexdigest()
    if not force_refresh and key in cache:
        return cache[key]

    context, ids = retrieve_chunks(collection, query, doc_type=doc_type)
    if not len(ids):
        result = {"answer": "No relevant info found.", "grounded": True, "chunks_used": []}
    else:
        answer = completion(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": ("Synthesize an answer using ONLY the provided context. "
                 "Cite the chunk ID in [brackets] after every claim. If info is missing, output "
                 "'INFO NOT IN CONTEXT'.")},
                {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUERY: {query}"},
            ],
            temperature=0.0, api_key=userdata.get('GEMINI_API_KEY'),
        ).choices[0].message.content
        v = verify_citations(answer, context)
        result = {"answer": answer, "grounded": v.get("grounded"),
                  "unsupported_claims": v.get("unsupported_claims", []), "chunks_used": list(ids)}

    cache[key] = result
    save_query_cache(query_cache_path, cache)
    return result

def ask_hybrid_fast(collection, G, query, resolved_entities, query_cache_path, force_refresh=False):
    cache = load_query_cache(query_cache_path)
    key = hashlib.md5(f"hybrid::{query}".encode()).hexdigest()
    if not force_refresh and key in cache:
        return cache[key]

    graph_facts = extract_graph_context(G, resolved_entities)
    vector_context, ids = retrieve_chunks(collection, query, doc_type=None)
    if not len(ids) and not graph_facts:
        result = {"answer": "No relevant info found.", "grounded": True, "chunks_used": [], "graph_facts_used": []}
    else:
        full_context = ("--- KNOWLEDGE GRAPH FACTS ---\n" + ("\n".join(graph_facts) or "None") +
                        "\n\n--- VECTOR CHUNKS ---\n" + vector_context)
        answer = completion(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": ("Synthesize an answer using ONLY the context. Cite vector "
                 "chunks as [chunk_id] and graph facts as [graph:relation]. If missing, say "
                 "'INFO NOT IN CONTEXT'.")},
                {"role": "user", "content": f"CONTEXT:\n{full_context}\n\nQUERY: {query}"},
            ],
            temperature=0.0, api_key=userdata.get('GEMINI_API_KEY'),
        ).choices[0].message.content
        v = verify_citations(answer, full_context)
        result = {"answer": answer, "grounded": v.get("grounded"),
                  "unsupported_claims": v.get("unsupported_claims", []),
                  "chunks_used": list(ids), "graph_facts_used": graph_facts}

    cache[key] = result
    save_query_cache(query_cache_path, cache)
    return result

# ---------------------------------------------------------
# 5. ENTITY RESOLUTION
# ---------------------------------------------------------
def fast_entity_resolution(query, collection):
    found = set()
    for eq in KNOWN_EQUIPMENTS:
        if eq.lower() in query.lower():
            found.add(eq)
    for haz in KNOWN_HAZARDS:
        if haz.lower() in query.lower():
            found.add(haz)
    if found:
        return list(found)
    results = collection.query(query_texts=[query], n_results=2)
    if results and results.get('metadatas'):
        for meta in results['metadatas'][0]:
            eq = meta.get('equipment_id')
            if eq and eq != 'UNKNOWN' and eq in KNOWN_EQUIPMENTS:
                found.add(eq)
    return list(found)

# ---------------------------------------------------------
# 6. LANGGRAPH STATE & NODES
# ---------------------------------------------------------
def merge_dicts(a: dict, b: dict) -> dict:
    c = a.copy()
    c.update(b)
    return c

class AgentState(TypedDict):
    query: str
    resolved_entities: List[str]
    intents: List[str]
    specialist_outputs: Annotated[Dict[str, Any], merge_dicts]
    final_answer: str

def preflight_classifier_node(state: AgentState):
    query = state["query"]
    entities = fast_entity_resolution(query, db_collection)
    prompt = f"""Analyze this query: "{query}"\nPre-detected entities (via deterministic matching): {entities}\n\nTask 1: Classify intent into one or more of: ["regulatory", "operational", "relational", "unsupported"].\nTask 2: If no entity was pre-detected, check if the query indirectly references any equipment\nfrom {KNOWN_EQUIPMENTS} or hazards from {KNOWN_HAZARDS} (e.g. by symptom or context), and extract it.\n\nReturn strict JSON only:\n{{"intents": ["..."], "entities": ["..."]}}"""
    resp = completion(model=MODEL_NAME, messages=[{"role": "user", "content": prompt}],
                        temperature=0.0, api_key=userdata.get('GEMINI_API_KEY'))
    raw = re.sub(r'^```json\s*|\s*```$', '', resp.choices[0].message.content.strip())
    try:
        data = json.loads(raw)
        intents = data.get("intents") or ["relational"]
        combined = list(set(entities + data.get("entities", [])))
    except json.JSONDecodeError:
        intents, combined = ["relational"], entities

    print(f"🔍 Resolved Entities: {combined}")
    print(f"🚦 Classification Intents: {intents}")
    return {"resolved_entities": combined, "intents": intents}

def regulatory_agent_node(state: AgentState):
    print("   -> 🏛️ Regulatory Specialist")
    return {"specialist_outputs": {"regulatory": ask_fast(db_collection, state["query"], query_cache_path, doc_type="regulatory")}}

def operational_agent_node(state: AgentState):
    print("   -> 🔧 Operational Specialist")
    query = state["query"]

    # Extract the first known equipment to use as a hard metadata filter
    resolved_equipment = [e for e in state["resolved_entities"] if e in KNOWN_EQUIPMENTS]
    eq_filter = resolved_equipment[0] if resolved_equipment else None

    best_cat, best_context, best_ids = None, "", []
    for cat in ["maintenance_logs", "incident_reports", "work_permits"]:
        # Build the where clause injecting the equipment ID if we found one
        where_clause = {"$and": [{"doc_type": cat}, {"equipment_id": eq_filter}]} if eq_filter else {"doc_type": cat}

        results = db_collection.query(query_texts=[query], n_results=5, where=where_clause)

        if results and results['ids'] and len(results['ids'][0]) > 0:
            ids = results['ids'][0]
            context = "\n\n".join(f"--- ID: {cid} ---\n{txt}" for cid, txt in zip(ids, results['documents'][0]))
            if len(ids) > len(best_ids):
                best_cat, best_context, best_ids = cat, context, ids

    if not best_ids:
        result = {"answer": "No relevant info found.", "grounded": True, "chunks_used": []}
    else:
        answer = completion(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": ("Synthesize an answer using ONLY the provided context. "
                 "Cite the chunk ID in [brackets] after every claim. If info is missing, output "
                 "'INFO NOT IN CONTEXT'.")},
                {"role": "user", "content": f"CONTEXT:\n{best_context}\n\nQUERY: {query}"},
            ],
            temperature=0.0, api_key=userdata.get('GEMINI_API_KEY')
        ).choices[0].message.content
        v = verify_citations(answer, best_context)
        result = {"answer": answer, "grounded": v.get("grounded"),
                   "unsupported_claims": v.get("unsupported_claims", []), "chunks_used": list(best_ids)}

    return {"specialist_outputs": {"operational": result}}

def relational_agent_node(state: AgentState):
    print("   -> 🕸️ Relational Specialist")
    result = ask_hybrid_fast(db_collection, KG, state["query"], state["resolved_entities"], query_cache_path)
    return {"specialist_outputs": {"relational": result}}

def synthesizer_node(state: AgentState):
    if "unsupported" in state["intents"] and not state["specialist_outputs"]:
        return {"final_answer": "I can only answer questions related to industrial safety, equipment maintenance, and regulations."}

    outputs = state["specialist_outputs"]
    if len(outputs) == 1:
        return {"final_answer": list(outputs.values())[0]["answer"]}

    merged_context = "\n\n".join(f"--- Agent ({k}) ---\n{v['answer']}" for k, v in outputs.items())
    prompt = f"Synthesize a single answer for '{state['query']}' from these outputs, keeping exact citations:\n\n{merged_context}"
    ans = completion(model=MODEL_NAME, messages=[{"role": "user", "content": prompt}],
                      temperature=0.0, api_key=userdata.get('GEMINI_API_KEY')).choices[0].message.content
    v = verify_citations(ans, merged_context)
    if not v.get("grounded", True):
        print(f"⚠️  Merged answer has unsupported claims: {v.get('unsupported_claims')}")
    return {"final_answer": ans}

def route_logic(state: AgentState):
    intents = list(state["intents"])
    if "unsupported" in intents:
        return ["synthesizer"]

    # Safety net: if a specific equipment is resolved and this isn't purely
    # operational/regulatory in isolation, force relational reasoning in too —
    # don't rely on the classifier remembering this every time.
    resolved_equipment = [e for e in state["resolved_entities"] if e in KNOWN_EQUIPMENTS]
    if resolved_equipment and ("regulatory" in intents or "operational" in intents) and "relational" not in intents:
        intents.append("relational")
        print(f"   ⚙️ Safety net: forcing 'relational' — equipment {resolved_equipment} resolved alongside {intents}")

    dest = []
    if "regulatory" in intents:
        dest.append("regulatory_agent")
    if "operational" in intents:
        dest.append("operational_agent")
    if "relational" in intents:
        dest.append("relational_agent")

    return dest or ["synthesizer"]

builder = StateGraph(AgentState)
builder.add_node("preflight", preflight_classifier_node)
builder.add_node("regulatory_agent", regulatory_agent_node)
builder.add_node("operational_agent", operational_agent_node)
builder.add_node("relational_agent", relational_agent_node)
builder.add_node("synthesizer", synthesizer_node)
builder.add_edge(START, "preflight")
builder.add_conditional_edges("preflight", route_logic,
                                ["regulatory_agent", "operational_agent", "relational_agent", "synthesizer"])
builder.add_edge("regulatory_agent", "synthesizer")
builder.add_edge("operational_agent", "synthesizer")
builder.add_edge("relational_agent", "synthesizer")
builder.add_edge("synthesizer", END)
orchestrator_graph = builder.compile()

# ---------------------------------------------------------
# 7. QUERY-LEVEL CACHE
# ---------------------------------------------------------
def load_query_cache(path):
    return json.load(open(path, encoding='utf-8')) if os.path.exists(path) else {}

def save_query_cache(path, cache):
    json.dump(cache, open(path, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

def ask_orchestrator(query: str):
    print(f"\n{'='*80}\n🤖 ORCHESTRATOR QUERY: {query}\n{'='*80}")
    cache = load_query_cache(query_cache_path)
    key = f"orchestrated::{hashlib.md5(query.encode()).hexdigest()}"

    is_cached = key in cache
    if is_cached:
        r = cache[key]
        print("⚡ [Cache Hit] — 0 LLM calls made.")
        print(f"🔍 Resolved Entities: {r['resolved_entities']}")
        print(f"🚦 Router Intents: {r['intents']}")
        print(f"✅ Final Answer: {r['final_answer']}")
        return r, True  # Returns True indicating it was a cache hit

    try:
        result_state = orchestrator_graph.invoke(
            {"query": query, "resolved_entities": [], "intents": [], "specialist_outputs": {}}
        )
    except Exception as e:
        print(f"🛑 Query failed: {e}")
        return {"query": query, "error": str(e)}, False

    print(f"\n✅ Final Answer:\n{result_state['final_answer']}")
    cache[key] = {
        "query": query,
        "resolved_entities": result_state["resolved_entities"],
        "intents": result_state["intents"],
        "final_answer": result_state["final_answer"],
        "specialist_outputs": result_state["specialist_outputs"],
    }
    save_query_cache(query_cache_path, cache)
    return cache[key], False  # Returns False indicating it was a fresh API execution

# ---------------------------------------------------------
# 8. INITIALIZATION
# ---------------------------------------------------------
ensure_drive_mounted()
base_path = "/content/drive/MyDrive/ET_Industrial_data"
verify_base_path(base_path)

data_dirs = [os.path.join(base_path, "regulatory"), os.path.join(base_path, "synthetic")]
cache_path = os.path.join(base_path, "master_chunks.json")
chroma_path = os.path.join(base_path, "vectorstore")
json_path = os.path.join(base_path, "synthetic", "industrial_records.json")
graph_path = os.path.join(base_path, "knowledge_graph.pkl")
query_cache_path = os.path.join(base_path, "query_cache.json")
os.makedirs(chroma_path, exist_ok=True)

print("\n🚀 Loading Vector Database & Knowledge Graph...")
db_collection = load_vector_store(data_dirs, cache_path, chroma_path)
KG = load_knowledge_graph(json_path, graph_path)
print(f"✅ Ready for Phase 4 | DB chunks: {db_collection.count()} | KG nodes: {KG.number_of_nodes()}")

# ---------------------------------------------------------
# 9. TEST QUERIES
# ---------------------------------------------------------
if __name__ == "__main__":
    tests = [
        "What safety measures apply to fencing of machinery?",
        "What is the maintenance history of PUMP-14?",
        "What permits were active for confined space work, and what regulation applies?",
        "Tell me about the pump that had a bearing failure and whether the Factory Act applies to it",
        "what's the compensation payout for a level-4 fire injury",
    ]
    for i, t in enumerate(tests):
        _, cache_hit = ask_orchestrator(t)
        if i < len(tests) - 1:
            if cache_hit:
                print("\n⚡ [Cache Hit] Skipping cooldown entirely (0s wait).")
            else:
                print("\n⏳ Cooling down for 80s before next query (rate limit safety)...")
                time.sleep(80)

✅ Google Drive verified as mounted at /content/drive
✅ base_path verified — 5 PDF(s) found

🚀 Loading Vector Database & Knowledge Graph...
✅ Vector store already current.
📊 Total chunks in DB: 513
✅ Knowledge graph loaded from cache.
✅ Ready for Phase 4 | DB chunks: 513 | KG nodes: 85

🤖 ORCHESTRATOR QUERY: What safety measures apply to fencing of machinery?
⚡ [Cache Hit] — 0 LLM calls made.
🔍 Resolved Entities: ['Mechanical']
🚦 Router Intents: ['regulatory']
✅ Final Answer: Based on the provided context, the safety measures that apply to the fencing of machinery in a factory include the following:

* **General Requirement:** Machinery must be securely fenced by safeguards of substantial construction. These safeguards must be constantly maintained and kept in position while the parts of the machinery they are fencing are in motion or in use [chunk_78].
* **Specific Parts Requiring Fencing:** The following machinery parts must be securely fenced:
  * Every moving part of a prime mover a

18:35:49 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1104 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.5-flash) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

🛑 Query failed: litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
    "status": "UNAVAILABLE"
  }
}


⏳ Cooling down for 80s before next query (rate limit safety)...


KeyboardInterrupt: 

In [ ]:
def dry_run_query(query):
    """Zero LLM calls. Shows exactly what would be fed into generation,
    so you can paste this to Claude for a correctness check before
    spending any API quota on it."""
    print(f"\n{'='*80}\n🔬 DRY RUN: {query}\n{'='*80}")
    entities = fast_entity_resolution(query, db_collection)
    print(f"🔍 Resolved Entities: {entities}")

    print("\n--- Regulatory ---")
    ctx, ids = retrieve_chunks(db_collection, query, doc_type="regulatory")
    print(f"IDs: {list(ids)}")

    print("\n--- Operational (per category, equipment-filtered if resolved) ---")
    eq = [e for e in entities if e in KNOWN_EQUIPMENTS]
    eq_filter = eq[0] if eq else None
    for cat in ["maintenance_logs", "incident_reports", "work_permits"]:
        where = {"$and": [{"doc_type": cat}, {"equipment_id": eq_filter}]} if eq_filter else {"doc_type": cat}
        r = db_collection.query(query_texts=[query], n_results=5, where=where)
        for cid, txt in zip(r['ids'][0], r['documents'][0]):
            print(f"  [{cat}] {cid}: {txt[:150]}")

    print("\n--- Graph facts ---")
    for f in extract_graph_context(KG, entities):
        print(" ", f)

    return entities

In [ ]:
dry_run_query("Tell me about the pump that had a bearing failure and whether the Factory Act applies to it")
dry_run_query("what's the compensation payout for a level-4 fire injury")


🔬 DRY RUN: Tell me about the pump that had a bearing failure and whether the Factory Act applies to it
🔍 Resolved Entities: ['PUMP-14']

--- Regulatory ---
IDs: ['chunk_221', 'chunk_51', 'chunk_86', 'chunk_110', 'chunk_186']

--- Operational (per category, equipment-filtered if resolved) ---
  [maintenance_logs] maintenance_logs_M-101: Record Type: maintenance_logs
log_id: M-101
equipment_id: PUMP-14
date: 2026-06-10
issue: Severe vibration and bearing noise.
resolution: Replaced thr
  [maintenance_logs] maintenance_logs_M-115: Record Type: maintenance_logs
log_id: M-115
equipment_id: PUMP-14
date: 2026-07-16
issue: Casing temperature high.
resolution: Cleared blockage in suc
  [maintenance_logs] maintenance_logs_M-107: Record Type: maintenance_logs
log_id: M-107
equipment_id: PUMP-14
date: 2026-06-28
issue: Minor seal leak detected.
resolution: Replaced mechanical se
  [incident_reports] incident_reports_IR-201: Record Type: incident_reports
incident_id: IR-201
equipment_id: PUMP-14


[]

In [ ]:
"""
Phase 5 — Gradio interface for the Industrial Knowledge Intelligence platform.

Two modes:
  MOCK_MODE = True   -> uses canned responses (for local smoke-testing only,
                         no API key or Drive needed — this is how it was
                         verified to actually run before being handed over)
  MOCK_MODE = False  -> calls your real ask_orchestrator() from Phase 4.
                         Paste your Phase 4 script's functions above this
                         file (or import them) before flipping this switch.
"""

import gradio as gr

MOCK_MODE = True  # flip to False once wired to your real Phase 4 orchestrator

# ---------------------------------------------------------
# REAL BACKEND HOOK — replace this function's body with a call to your
# actual Phase 4 ask_orchestrator(query) once MOCK_MODE = False
# ---------------------------------------------------------
def real_ask(query: str) -> dict:
    # from phase4_orchestrator import ask_orchestrator
    # result, _ = ask_orchestrator(query)
    # return result
    raise NotImplementedError("Wire this to your real Phase 4 orchestrator import.")

# ---------------------------------------------------------
# MOCK BACKEND — canned answers matching the real logged outputs from
# Phase 2-4 testing, used only to verify the interface itself works
# ---------------------------------------------------------
_MOCK_RESPONSES = {
    "fencing": {
        "answer": ("Machinery must be securely fenced by safeguards of substantial "
                   "construction, including every moving part of a prime mover and "
                   "flywheel [chunk_78]."),
        "intents": ["regulatory"], "entities": [],
    },
    "pump-14": {
        "answer": ("PUMP-14's maintenance history includes severe vibration/bearing "
                   "noise resolved by thrust bearing replacement [maintenance_logs_M-101], "
                   "a seal leak [maintenance_logs_M-107], and a casing temperature issue "
                   "[maintenance_logs_M-115]."),
        "intents": ["operational", "relational"], "entities": ["PUMP-14"],
    },
    "compensation": {
        "answer": "INFO NOT IN CONTEXT",
        "intents": ["unsupported"], "entities": [],
    },
}

def mock_ask(query: str) -> dict:
    q = query.lower()
    if "fenc" in q:
        return _MOCK_RESPONSES["fencing"]
    if "pump-14" in q or "bearing" in q:
        return _MOCK_RESPONSES["pump-14"]
    if "compensation" in q or "payout" in q:
        return _MOCK_RESPONSES["compensation"]
    return {"answer": "INFO NOT IN CONTEXT", "intents": ["unsupported"], "entities": []}

def get_backend():
    return mock_ask if MOCK_MODE else real_ask

# ---------------------------------------------------------
# GRADIO UI
# ---------------------------------------------------------
def respond(message, history):
    backend = get_backend()
    result = backend(message)
    answer = result.get("answer", "No answer returned.")
    intents = result.get("intents", [])
    entities = result.get("entities", [])
    meta = f"\n\n---\n*Routed as: {intents} | Entities: {entities or 'none'}*"
    return answer + meta

with gr.Blocks(title="Industrial Knowledge Intelligence") as demo:
    gr.Markdown("# 🏭 Industrial Knowledge Intelligence Platform")
    gr.Markdown(
        "Ask about safety regulations, equipment maintenance history, permits, "
        "or how they relate. " + ("**[MOCK MODE — demo responses only]**" if MOCK_MODE else "")
    )
    chatbot = gr.ChatInterface(
        fn=respond,
        examples=[
            "What safety measures apply to fencing of machinery?",
            "What is the maintenance history of PUMP-14?",
            "What permits were active for confined space work, and what regulation applies?",
            "Tell me about the pump that had a bearing failure and whether the Factory Act applies to it",
            "What's the compensation payout for a level-4 fire injury?",
        ],
    )

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f336535cbcb9a3aa1b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
"""
Phase 5 — Gradio interface, restyled to match the dark 'Aether' chat design
(colors/spacing pulled from the Figma export: indigo->violet gradient accents,
obsidian background, asymmetric bubble corners, subtle borders).

Same MOCK_MODE / real_ask() hook as before — only the presentation layer changed.
"""

import re
import gradio as gr

MOCK_MODE = True  # flip to False once wired to your real Phase 4 orchestrator

# ---------------------------------------------------------
# REAL BACKEND HOOK
# ---------------------------------------------------------
def real_ask(query: str) -> dict:
    # from phase4_orchestrator import ask_orchestrator
    # result, _ = ask_orchestrator(query)
    # return result
    raise NotImplementedError("Wire this to your real Phase 4 orchestrator import.")

# ---------------------------------------------------------
# MOCK BACKEND (unchanged from before)
# ---------------------------------------------------------
_MOCK_RESPONSES = {
    "fencing": {
        "answer": ("Machinery must be securely fenced by safeguards of substantial "
                   "construction, including every moving part of a prime mover and "
                   "flywheel [chunk_78]."),
        "intents": ["regulatory"], "entities": [],
    },
    "pump-14": {
        "answer": ("PUMP-14's maintenance history includes:\n\n"
                   "**Severe vibration/bearing noise** — resolved by thrust bearing "
                   "replacement [maintenance_logs_M-101]\n\n"
                   "**Minor seal leak** — resolved by mechanical seal replacement "
                   "[maintenance_logs_M-107]\n\n"
                   "**High casing temperature** — resolved by clearing a suction "
                   "strainer blockage [maintenance_logs_M-115]\n\n"
                   "The **Factories Act, 1948** applies to PUMP-14 via its Mechanical "
                   "and High Pressure hazard tags [graph:APPLIES_TO]."),
        "intents": ["operational", "relational"], "entities": ["PUMP-14"],
    },
    "compensation": {
        "answer": "INFO NOT IN CONTEXT",
        "intents": ["unsupported"], "entities": [],
    },
}

def mock_ask(query: str) -> dict:
    q = query.lower()
    if "fenc" in q:
        return _MOCK_RESPONSES["fencing"]
    if "pump-14" in q or "bearing" in q:
        return _MOCK_RESPONSES["pump-14"]
    if "compensation" in q or "payout" in q:
        return _MOCK_RESPONSES["compensation"]
    return {"answer": "INFO NOT IN CONTEXT", "intents": ["unsupported"], "entities": []}

def get_backend():
    return mock_ask if MOCK_MODE else real_ask

# ---------------------------------------------------------
# CITATION FORMATTING — the actual fix for the raw-bracket problem.
# Turns [chunk_78] / [maintenance_logs_M-101] / [graph:APPLIES_TO] into
# small styled pill badges instead of plain debug-looking brackets.
# ---------------------------------------------------------
_CITATION_PATTERN = re.compile(r'\[(chunk_\d+|[\w_]+_[\w\-]+|graph:[\w_]+)\]')

def _citation_label(raw: str) -> str:
    """Human-friendly label for a citation tag."""
    if raw.startswith("graph:"):
        return f"🕸 {raw.split(':', 1)[1]}"
    if raw.startswith("chunk_"):
        return f"📄 {raw}"
    # e.g. maintenance_logs_M-101 -> "🔧 M-101"
    parts = raw.rsplit("_", 1)
    icon = {"maintenance_logs": "🔧", "incident_reports": "⚠️", "work_permits": "📋"}.get(
        parts[0], "🔗"
    )
    return f"{icon} {parts[-1]}" if len(parts) == 2 else f"🔗 {raw}"

def format_answer(answer: str) -> str:
    def replace(match):
        raw = match.group(1)
        label = _citation_label(raw)
        return (
            f'<span style="display:inline-block;padding:1px 8px;margin:0 2px;'
            f'border-radius:999px;font-size:11px;font-weight:500;'
            f'background:rgba(99,102,241,0.15);border:1px solid rgba(99,102,241,0.35);'
            f'color:rgba(199,197,255,0.95);white-space:nowrap;">{label}</span>'
        )
    return _CITATION_PATTERN.sub(replace, answer)

_INTENT_COLORS = {
    "regulatory": "#8b5cf6",
    "operational": "#22c55e",
    "relational": "#6366f1",
    "unsupported": "#ef4444",
}

def format_meta(intents, entities) -> str:
    chips = "".join(
        f'<span style="display:inline-block;padding:2px 10px;margin-right:6px;'
        f'border-radius:999px;font-size:10px;font-weight:600;letter-spacing:.02em;'
        f'text-transform:uppercase;background:{_INTENT_COLORS.get(i, "#666")}22;'
        f'border:1px solid {_INTENT_COLORS.get(i, "#666")}55;'
        f'color:{_INTENT_COLORS.get(i, "#999")};">{i}</span>'
        for i in intents
    )
    ent_str = ", ".join(entities) if entities else "none"
    return (
        f'<div style="margin-top:10px;padding-top:8px;border-top:1px solid rgba(255,255,255,0.08);">'
        f'{chips}'
        f'<span style="color:rgba(255,255,255,0.3);font-size:11px;margin-left:4px;">'
        f'entities: {ent_str}</span></div>'
    )

def respond(message, history):
    backend = get_backend()
    result = backend(message)
    answer = format_answer(result.get("answer", "No answer returned."))
    meta = format_meta(result.get("intents", []), result.get("entities", []))
    return answer + meta

# ---------------------------------------------------------
# DARK THEME — colors/spacing lifted from the Figma export
# ---------------------------------------------------------
CUSTOM_CSS = """
.gradio-container {
    background: #0d0f14 !important;
}
#chatbot {
    background: transparent !important;
    border: none !important;
}
.message-wrap {
    gap: 20px !important;
}
.message.user {
    background: linear-gradient(135deg, #6366f1 0%, #7c3aed 100%) !important;
    border-radius: 16px 16px 4px 16px !important;
    border: none !important;
    color: rgba(255,255,255,0.95) !important;
}
.message.bot {
    background: rgba(255,255,255,0.06) !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    border-radius: 16px 16px 16px 4px !important;
    color: rgba(255,255,255,0.85) !important;
}
#input-row textarea {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    color: rgba(255,255,255,0.85) !important;
    border-radius: 16px !important;
}
.header-title {
    background: linear-gradient(135deg, #6366f1 0%, #8b5cf6 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-weight: 700;
}
"""

with gr.Blocks(title="Industrial Knowledge Intelligence", css=CUSTOM_CSS, theme=gr.themes.Base(
    primary_hue="indigo", neutral_hue="slate",
)) as demo:
    gr.Markdown(
        '<h1 class="header-title">🏭 Industrial Knowledge Intelligence</h1>\n\n'
        "Ask about safety regulations, equipment maintenance history, permits, or how they relate."
        + (" **[MOCK MODE — demo responses only]**" if MOCK_MODE else "")
    )
    gr.ChatInterface(
        fn=respond,
        chatbot=gr.Chatbot(elem_id="chatbot", height=480, render_markdown=True, sanitize_html=False),
        examples=[
            "What safety measures apply to fencing of machinery?",
            "What is the maintenance history of PUMP-14?",
            "What permits were active for confined space work, and what regulation applies?",
            "Tell me about the pump that had a bearing failure and whether the Factory Act applies to it",
            "What's the compensation payout for a level-4 fire injury?",
        ],
    )

if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_31183/2656614306.py:167: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Industrial Knowledge Intelligence", css=CUSTOM_CSS, theme=gr.themes.Base(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://001b6c4421ba18f501.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
